In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.1.5: Fast Llama patching. Transformers: 4.47.1.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.3, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.3.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.1.5 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
!pip install datasets

### custom dataset or use this dataset https://huggingface.co/datasets/peteparker456/english-to-tanglish-translation-dataset

In [ ]:
from datasets import Dataset

# Assuming the dataset has already been loaded
json_file_path = "/content/filtered_translation_data.json"  # Replace with the path to your JSON file
with open(json_file_path, "r") as f:
    custom_data = json.load(f)

# Convert the JSON data to a Hugging Face Dataset
dataset = Dataset.from_list(custom_data)

# Split the dataset into train and test sets (e.g., 80% train, 20% test)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Access the train and test subsets
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

# Print the sizes of the train and test datasets
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


In [ ]:
#from datasets import load_dataset
#
#ds = load_dataset(
#    "peteparker456/english-to-tanglish-translation-dataset"
#)


In [ ]:
dataset.features,dataset[0]

({'human': Value(dtype='string', id=None),
  'assistant': Value(dtype='string', id=None)},
 {'human': 'one', 'assistant': 'one'})

In [ ]:
# Define the Alpaca-style prompt
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the given English text into colloquial Tamil.

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Ensure EOS_TOKEN is used

# Function to format prompts for your dataset
def formatting_prompts_func(examples):
    humans = examples["human"]
    assistants = examples["assistant"]
    texts = []
    for human, assistant in zip(humans, assistants):
        # Format each example using the Alpaca prompt
        text = alpaca_prompt.format("Translate to the assistant's language.", human, assistant) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply the formatting function to your dataset
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# View the formatted dataset
print(formatted_dataset)

Map:   0%|          | 0/618 [00:00<?, ? examples/s]

Dataset({
    features: ['human', 'assistant', 'text'],
    num_rows: 618
})


<a name="Train"></a>
### Train the model


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Adjust for shorter runs.
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",  # Use this for WandB etc.
        load_best_model_at_end=True,  # Load the best model at the end.
        metric_for_best_model="eval_loss",  # Use the evaluation loss as the metric.
        greater_is_better=False,  # Lower eval loss is better.
        save_total_limit=2,  # Keep only the 2 most recent checkpoints.
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Stop if no improvement after 3 checks.
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=128,  # Adjust based on the average length of your dataset samples
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Keep small to fit into T4 GPU memory
        gradient_accumulation_steps=8,  # Simulate a larger effective batch size
        warmup_steps=20,  # Higher warmup steps to stabilize training
        num_train_epochs=10,  # Increase epochs for small datasets
        max_steps=-1,  # Train for full epochs instead of limiting steps
        learning_rate=1e-4,  # Slightly lower learning rate to avoid overfitting
        fp16=True,  # Enable mixed precision for memory efficiency
        logging_steps=10,  # Log every 10 steps
        optim="adamw_torch",  # Torch implementation can be more efficient
        weight_decay=0.01,
        lr_scheduler_type="cosine",  # Use cosine decay for smoother scheduling
        seed=42,  # Set for reproducibility
        output_dir="outputs",
        save_steps=50,  # Save checkpoint every 50 steps
        evaluation_strategy="no",  # No evaluation since it's fine-tuning
        save_total_limit=2,  # Keep only 2 checkpoints to save space
        report_to="none",
    ),
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Map (num_proc=2):   0%|          | 0/618 [00:00<?, ? examples/s]

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
5.848 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 618 | Num Epochs = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 8
\        /    Total batch size = 16 | Total steps = 380
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
10,2.588500
20,1.438500
30,0.383000
40,0.330500
50,0.264200
60,0.202800
70,0.200300
80,0.189000
90,0.179700
100,0.162100


Step,Training Loss
10,2.588500
20,1.438500
30,0.383000
40,0.330500
50,0.264200
60,0.202800
70,0.200300
80,0.189000
90,0.179700
100,0.162100


In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

5699.2384 seconds used for training.
94.99 minutes used for training.
Peak reserved memory = 6.631 GB.
Peak reserved memory for training = 0.783 GB.
Peak reserved memory % of max memory = 44.962 %.
Peak reserved memory for training % of max memory = 5.309 %.


<a name="Inference"></a>
### Inference


In [ ]:
from IPython.display import display, Markdown

FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [
        prompt_style.format(
            "Translate the following English sentence into colloquial Tamil or Tanglish: 'hey buddy we won the match'",
            "",
        )
    ],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=250,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
Markdown(response[0].split("\n\n### Response:")[1])


என்னட buddy நாம match வென்றோம்<|end_of_text|>

In [ ]:
from huggingface_hub import notebook_login   ###  can get the hugging face api token in theor official website
notebook_login()

In [ ]:
model.push_to_hub("peteparker456/llama-unsloth") # Online saving
tokenizer.push_to_hub("peteparker456/llama-unsloth") # Online saving

README.md:   0%|          | 0.00/594 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/peteparker456/llama-unsloth


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')